In [10]:
!cat lineup-classes.nomnoml | grep -P "<:-" | head -n 2

[ILineUpOptions]<:--[ITaggleOptions]
[LineUpBuilder]<:-[DataBuilder]
grep: write error: Broken pipe


In [44]:
import re
from collections import Counter

# ---------- Step 1: Parse associations (unchanged) ----------
association_freq = Counter()
entity_type_freq = Counter()
connected_items = set()

associations = (
    r"\[([^\]]+)\]\s*"
    r"(<-->|--:>|<:--|<->|-->|-:>|<:-|\+->|o->|-/-|->|\+-|o-|--|_>|__|-)"
    r"\s*\[([^\]]+)\]"
)
assoc_pattern = re.compile(associations)

with open('lineup-classes.nomnoml') as f:
    content = f.read()

for match in assoc_pattern.finditer(content):
    left, arrow, right = match.groups()
    association_freq[arrow] += 1
    connected_items.add(left.strip())
    connected_items.add(right.strip())

# ---------- Step 2: Parse entity definitions, LINE BY LINE ----------
# Only need the name here -- match from start of line, stop at first
# "|" or "<" (generic) which always appears right after the bare name.
name_pattern = re.compile(
    r"^\s*\[(?:<(\w+)>)?([A-Za-z_]\w*)"
)

entities = []  # (full_line_text, bare_name)
with open('lineup-classes.nomnoml') as f:
    lines = f.readlines()

for line in lines:
    if "|" not in line:
        continue  # not a definition line (no attributes/methods pipe)
    m = name_pattern.match(line)
    if m:
        stereotype, name = m.groups()
        entity_kind = stereotype if stereotype else "class"
        entities.append((line, name, entity_kind))
        entity_type_freq[entity_kind] += 1

# ---------- Step 3: Find unconnected entities ----------
unconnected = [
    (full_line, name, kind) for full_line, name, kind in entities
    if name not in connected_items
]

print(f"Found {len(entities)} entity definitions.")
print(f"Found {len(unconnected)} unconnected entities:")
for _, name, kind in unconnected:
    print(f"  - {name} ({kind})")

# ---------- Step 4: Remove unconnected lines, write edited.nomnoml ----------
unconnected_lines = {full_line for full_line, _, _ in unconnected}

new_lines = [line for line in lines if line not in unconnected_lines]

with open('edited.nomnoml', 'w') as f:
    f.writelines(new_lines)

print(f"\nWrote edited.nomnoml with {len(unconnected)} entities removed.")

Found 403 entity definitions.
Found 181 unconnected entities:
  - IDynamicHeight (interface)
  - ILineUpFlags (interface)
  - IToolbarLookup (interface)
  - ILivePreviewOptions (interface)
  - RankingBuilder (class)
  - IImposeColumnBuilder (interface)
  - INestedBuilder (interface)
  - IWeightedSumBuilder (interface)
  - IReduceBuilder (interface)
  - IScriptedBuilder (interface)
  - IEventContext (interface)
  - IEventListener (interface)
  - IDebounceContext (interface)
  - IDragStartResult (interface)
  - IDropResult (interface)
  - IDragHandleOptions (interface)
  - LazyFilter (class)
  - ALazyMap (class)
  - LazyMap1 (class)
  - LazyMap2 (class)
  - LazyMap3 (class)
  - LazySeq (class)
  - ConcatSequence (class)
  - IForEachAble (interface)
  - ISequence (interface)
  - IBuilder (interface)
  - ISortMessageRequest (interface)
  - ISortMessageResponse (interface)
  - IDeleteRefMessageRequest (interface)
  - ISetRefMessageRequest (interface)
  - IDateStatsMessageRequest (interface)

In [23]:
association_types = {"<:-" : "generalization", "<:--" : "implementation"}
for a in association_freq:
  print(association_types[a], association_freq[a])

implementation 155
generalization 104


In [45]:
for e, c in entity_type_freq.items():
  print(e, c)

interface 210
class 181
enumeration 12


**NOTE**: Because we know that the class diagram has only two associations, we can try to compute some metadata about the diagram, such as 
* Classes that are being extended the most
* Interfaces that are being implemented the most
* Multi-class inheritance (Is possible but uncommon)
* Multi-interface implementations (Is possible but uncommon)

In [40]:
import re
from collections import Counter

association_freq = Counter()
connected_items = set()

# how many subclasses extend this entity (it's the base/parent)
extended_count = Counter()
# how many classes implement this entity (it's the interface)
implemented_count = Counter()
# how many base classes this entity extends (should be 0 or 1 — single inheritance)
extends_count = Counter()
# how many interfaces this entity implements (commonly > 1)
implements_count = Counter()

association_types = {"<:-": "generalization", "<:--": "implementation"}

associations = (
    r"\[([^\]]+)\]\s*"
    r"(<:--|<:-)"
    r"\s*\[([^\]]+)\]"
)
assoc_pattern = re.compile(associations)

with open('lineup-classes.nomnoml') as f:
    content = f.read()

for match in assoc_pattern.finditer(content):
    left, arrow, right = match.groups()
    left, right = left.strip(), right.strip()
    #print("[DEBUBG]", left, arrow, right)

    association_freq[arrow] += 1
    connected_items.add(left)
    connected_items.add(right)

    atype = association_types.get(arrow)

    # Arrowhead points left (<), so the SOURCE (subclass/implementer) is on
    # the right, and the TARGET (base class/interface) is on the left.
    if atype == "generalization":
        extends_count[right] += 1     # right extends left
        extended_count[left] += 1     # left is being extended
    elif atype == "implementation":
        implements_count[right] += 1  # right implements left
        implemented_count[left] += 1  # left is being implemented

# ---------- Most extended base classes (common) ----------
print("Base classes extended by multiple subclasses:")
for name, count in extended_count.most_common(10):
    if count > 1:
        print(f"  {name}: {count}")

# ---------- Most implemented interfaces (common) ----------
print("\nInterfaces implemented by multiple classes:")
for name, count in implemented_count.most_common(10):
    if count > 1:
        print(f"  {name}: {count}")

# ---------- Classes extending multiple base classes (rare/unexpected) ----------
print("\n⚠️  Classes extending MULTIPLE base classes (unusual — check these):")
suspicious = {name: c for name, c in extends_count.items() if c > 1}
if suspicious:
    for name, count in Counter(suspicious).most_common():
        print(f"  {name}: extends {count} base classes")
else:
    print("  None found — as expected for single-inheritance languages.")

# ---------- Classes implementing multiple interfaces (normal) ----------
print("\nClasses implementing the most interfaces:")
for name, count in implements_count.most_common(10):
    print(f"  {name}: implements {count} interfaces")

Base classes extended by multiple subclasses:
  ADialog: 24
  ValueColumn<T>: 13
  AEventDispatcher: 10
  ColumnBuilder<T>: 7
  Column: 6
  ArrayColumn<T>: 6
  APopup: 6
  MapColumn<T>: 5
  CompositeColumn: 5
  ALazyMap<T,T2>: 3

Interfaces implemented by multiple classes:
  ICellRendererFactory: 36
  ISequence<T>: 7
  IBuilderAdapterColumnDescProps: 7
  IColorMappingFunction: 6
  INumberColumn: 6
  IArrayColumn<T>: 5
  ICategoricalColumn: 4
  Column: 4
  IMapAbleColumn: 4
  IColumnDesc: 3

⚠️  Classes extending MULTIPLE base classes (unusual — check these):
  None found — as expected for single-inheritance languages.

Classes implementing the most interfaces:
  IArrayColumnDesc<T>: implements 2 interfaces
  ISetColumn: implements 2 interfaces
  IDatesColumn: implements 2 interfaces
  ImpositionCompositeColumn: implements 2 interfaces
  IBoxPlotColumn: implements 2 interfaces
  INumbersColumn: implements 2 interfaces
  NumberColumn: implements 2 interfaces
  INumbersDesc: implements 2 